# 06. Memory Optimization: Selective Checkpointing, Offloading, and Grad Hooks

- **Category:** 12_compilation_and_performance
- **Lineage & Citations:** Chen et al. (2016 *Training Deep Nets with Sublinear Memory Cost*); Korthikanti et al. (2023 *Reducing Activation Recomputation in Large Transformer Models*); Ren et al. (2021 *ZeRO-Offload: Democratizing Billion-Scale Model Training*).
- **Purpose:** Formulate and evaluate memory optimization paradigms: selective activation checkpointing (saving high-FLOP GEMM outputs while discarding cheap non-linearities), CPU/pinned memory offloading, and post-accumulate gradient hooks for overlapping communication.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [05. Torch Library Custom Ops and Fake Kernels](05_torch_library_custom_ops_and_fake_kernels.ipynb) $\to$ `🔗 Downstream Syntheses:` [07. Profiling Roofline Analysis CUDA Graphs and Fused Optimizers](07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not recompute compute-heavy matrix multiplications (only recompute memory-heavy pointwise activations); do not offload activations to CPU if PCIe bandwidth is the primary bottleneck.

## Mathematical Foundations & Recomputation Calculus

1. **Selective Checkpoint Gated SiLU**:
   $$\mathbf{z} = \mathbf{x} \mathbf{W} + \mathbf{b}, \quad \mathbf{a} = \mathbf{z} \odot \sigma(\mathbf{z}), \quad \mathbf{y} = \mathbf{a} \odot \mathbf{g}$$
2. **Adjoint Gradient with On-The-Fly Recomputation**:
   $$\mathbf{g}_z = (\mathbf{g}_y \odot \mathbf{g}) \odot [\sigma(\mathbf{z}) \cdot (1 + \mathbf{z} \cdot (1 - \sigma(\mathbf{z})))]$$
   $$\nabla_{\mathbf{W}} = \mathbf{x}^T \mathbf{g}_z, \quad \nabla_{\mathbf{b}} = \sum \mathbf{g}_z, \quad \nabla_{\mathbf{x}} = \mathbf{g}_z \mathbf{W}^T, \quad \nabla_{\mathbf{g}} = \mathbf{g}_y \odot \mathbf{a}$$
3. **Post-Accumulate Hook Momentum Step**:
   $$\mathbf{m}_{\text{new}} = \beta \mathbf{m} + (1 - \beta) \nabla_{\mathbf{W}}, \quad \mathbf{W}_{\text{new}} = \mathbf{W} - \eta \mathbf{m}_{\text{new}}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/12_compilation_and_performance/06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def selective_checkpoint_silu_gate(x, w, b, gate, grad_out):
    z = x @ w + b
    sig_z = torch.sigmoid(z)
    a = z * sig_z
    y = a * gate
    g_a = grad_out * gate
    da_dz = sig_z * (1.0 + z * (1.0 - sig_z))
    g_z = g_a * da_dz
    return y, x.t() @ g_z, torch.sum(g_z, dim=0), g_z @ w.t(), grad_out * a

def post_accumulate_grad_hook_step(w, grad_w, running_m, beta=0.9, lr=0.01):
    m_new = float(beta) * running_m + (1.0 - float(beta)) * grad_w
    w_new = w - float(lr) * m_new
    return w_new, m_new

def memory_footprint_analysis(batch, seq_len, d_model, d_ffn):
    tokens = float(batch * seq_len)
    mem_full = tokens * (d_model + 2 * d_ffn + d_model) * 4.0
    mem_selective = tokens * (2 * d_model) * 4.0
    return mem_full, mem_selective, 1.0 - (mem_selective / mem_full)

In [ ]:
# Verify selective recomputation and gradient parity against golden checkpoint
chk_y, grad_w, grad_b, grad_x, grad_gate = selective_checkpoint_silu_gate(
    weights["chk_x"], weights["chk_w"], weights["chk_b"], weights["chk_gate"], weights["chk_grad_out"]
)
assert torch.max(torch.abs(chk_y - weights["chk_y_ref"])) < 1e-5
assert torch.max(torch.abs(grad_w - weights["chk_grad_w_ref"])) < 1e-5
assert torch.max(torch.abs(grad_b - weights["chk_grad_b_ref"])) < 1e-5
assert torch.max(torch.abs(grad_x - weights["chk_grad_x_ref"])) < 1e-5
assert torch.max(torch.abs(grad_gate - weights["chk_grad_gate_ref"])) < 1e-5

# Verify post-accumulate grad hook step
w_new, m_new = post_accumulate_grad_hook_step(
    weights["chk_w"], grad_w, weights["hook_running_m"], beta=0.9, lr=0.01
)
assert torch.max(torch.abs(w_new - weights["hook_w_new_ref"])) < 1e-5
assert torch.max(torch.abs(m_new - weights["hook_m_new_ref"])) < 1e-5
print("Selective checkpointing & Post-accumulate hook parity verified!")

In [ ]:
# Microbenchmark: Selective recomputation execution latency
x, w, b, gate, g = weights["chk_x"], weights["chk_w"], weights["chk_b"], weights["chk_gate"], weights["chk_grad_out"]
t0 = time.perf_counter()
for _ in range(500):
    _ = selective_checkpoint_silu_gate(x, w, b, gate, g)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Joint selective forward + recompute backward latency: {lat_us:.2f} us")

In [ ]:
# Theoretical Activation Memory Analysis for LLaMA-style 7B feed-forward layer
m_full, m_sel, savings = memory_footprint_analysis(batch=4, seq_len=2048, d_model=4096, d_ffn=11008)
print(f"Full activations: {m_full / 1e6:.1f} MB | Selective activations: {m_sel / 1e6:.1f} MB")
print(f"Theoretical activation memory reduction: {savings * 100:.1f}%")
assert savings > 0.65, "Selective checkpointing must yield >65% activation memory reduction!"

# Autograd equivalence check
x_dyn = weights["chk_x"].clone().requires_grad_(True)
w_dyn = weights["chk_w"].clone().requires_grad_(True)
b_dyn = weights["chk_b"].clone().requires_grad_(True)
gate_dyn = weights["chk_gate"].clone().requires_grad_(True)
y_dyn = (x_dyn @ w_dyn + b_dyn) * torch.sigmoid(x_dyn @ w_dyn + b_dyn) * gate_dyn
y_dyn.backward(weights["chk_grad_out"])
assert float(torch.max(torch.abs(grad_w - w_dyn.grad))) < 1e-5, "Grad W mismatch!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Selective Checkpointing** retains GEMM boundary tensors while discarding cheap pointwise activations, eliminating over 65% of activation memory with minimal compute cost.
2. **Activation Offloading** leverages asynchronous pinned host memory streams to trade PCIe bandwidth for expanded working set capacity.
3. **Post-Accumulate-Grad Hooks** initiate communication and optimizer updates immediately upon gradient realization, overlapping communication with remaining backward layers.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.md](../../markdown/12_compilation_and_performance/06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.md)
- **PyTorch 2.x Module:** [06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.py](../../pytorch2/12_compilation_and_performance/06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.py)
- **Functional JAX Module:** [06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.py](../../jax/12_compilation_and_performance/06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.py)
- **Native Rust Candle Track:** [06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.rs](../../candle/12_compilation_and_performance/06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.rs)
- **Hardware Kernel (CPU / WGSL):** [06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.rs](../../kernels/12_compilation_and_performance/06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.rs)
- **Golden SafeTensors:** [06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.safetensors](../../weights/12_compilation_and_performance/06_memory_optimization_selective_checkpointing_offloading_and_grad_hooks.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 05. Torch Library Custom Ops and Fake Kernels](05_torch_library_custom_ops_and_fake_kernels.ipynb) | [Master Index](../../README.md#12_compilation_and_performance) | [Next: 07. Profiling Roofline Analysis CUDA Graphs and Fused Optimizers](07_profiling_roofline_analysis_cuda_graphs_and_fused_optimizers.ipynb) &rarr;